# Visualization of Euclid VIS Image and DESI Spectrum

This notebook loads and visualizes the outputs generated by the `generate_euclid_vis.py` script.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from astropy.io import fits
from astropy.visualization import (astropy_mpl_style, ZScaleInterval, 
                                   LogStretch, ImageNormalize)
import os

plt.style.use(astropy_mpl_style)

In [ ]:
# Define paths
data_dir = '/Users/marchuertascompany/Documents/data/EUCLID/syntesizer'
image_path = os.path.join(data_dir, 'euclid_vis.fits')
spectrum_path = os.path.join(data_dir, 'desi_spectrum.fits')

## 1. Euclid VIS Image

We use a log-stretch normalization to better see the galaxy structure.

In [ ]:
if os.path.exists(image_path):
    with fits.open(image_path) as hdul:
        data = hdul[0].data
        header = hdul[0].header
        
    norm = ImageNormalize(data, interval=ZScaleInterval(), stretch=LogStretch())
    
    plt.figure(figsize=(8, 8))
    plt.imshow(data, cmap='magma', norm=norm, origin='lower')
    plt.colorbar(label='Flux')
    plt.title(f"Euclid VIS Mock - {header.get('OBJECT', 'Galaxy')}")
    plt.xlabel('Pixels')
    plt.ylabel('Pixels')
    plt.show()
else:
    print(f"Image not found at {image_path}")

## 2. DESI Mock Spectrum

Plotting the convolved and resampled spectrum.

In [ ]:
if os.path.exists(spectrum_path):
    with fits.open(spectrum_path) as hdul:
        spec_data = hdul[1].data
        spec_header = hdul[1].header
        
    plt.figure(figsize=(12, 5))
    plt.plot(spec_data['wavelength'], spec_data['flux'], color='black', lw=0.5)
    
    # Highlight some common lines if they are in range
    lines = {
        'H-alpha': 6564.6,
        '[OIII]': 5008.2,
        'H-beta': 4862.7,
        '[OII]': 3728.5
    }
    
    z = spec_header.get('REDSHIFT', 0.0)
    for name, lam in lines.items():
        obs_lam = lam * (1 + z)
        if spec_data['wavelength'].min() < obs_lam < spec_data['wavelength'].max():
            plt.axvline(obs_lam, color='red', linestyle='--', alpha=0.3)
            plt.text(obs_lam, plt.ylim()[1]*0.9, name, color='red', rotation=90, verticalalignment='top')

    plt.title(f"DESI Mock Spectrum (z={z:.3f})")
    plt.xlabel('Wavelength [Angstrom]')
    plt.ylabel('Flux [erg/s/cm^2/Hz]')
    plt.grid(True, alpha=0.3)
    plt.show()
else:
    print(f"Spectrum not found at {spectrum_path}")